# 06 TAG Builder — Temporal Assessment Graph

**Phase 4 — Sequential Learning Analytics**  
Research contract: `PHASE4_RESEARCH_CONTRACT_v1.md`  
Schema version: `tag_v1`  
Prerequisite: M2 (`05_sequence_dataset.ipynb`) must be validated and complete.

## Roles of the TAG

1. **Learning-path analysis** — descriptive visualization of how learners navigate tasks  
2. **Graph-derived feature generation** — structural features fed to LSTM/GRU models (M4)

The TAG is **not** a classifier in Phase 4.  
No AUC, F1, or predictive performance is reported here.

## Two graph levels

| Level | Scope | Purpose |
|---|---|---|
| Student-instance TAG | One directed graph per learner × task | Trace individual learning paths |
| Cohort transition TAG | Aggregated node/edge weights | Population-level transition patterns |

**Split rule:**  
- Predictive analysis uses **train split only** (no test leakage into graph statistics)  
- Full pilot graph uses train+test combined for descriptive analysis only

## Anti-leakage contract

Graph features must NOT include:  
`at_risk`, `total_2c3l_score`, `grade_letter`, `is_teacher_reviewed`,  
any post-cutoff event, any rubric criterion score.

> ⚠️ **TECHNICAL VALIDATION ONLY** — proxy_behavioral / pilot_only labels.

## 0. Configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, hashlib, warnings
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings('ignore', category=FutureWarning)

SEQ_DIR = Path('data/sequences')
TAG_DIR = Path('data/tag')
REP_DIR = Path('../reports/phase4')
TAG_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_VERSION         = 'tag_v1'
M2_SCHEMA_VERSION      = 'seq_v1'
PHASE3_SOURCE_SHA      = '193b18949e40e6bd3bbfb70034a5772ce51d1b7e'

# Transition types (research contract §6)
TT_NEXT_EVENT           = 'NEXT_EVENT'
TT_RETRY                = 'RETRY'
TT_REVISION             = 'REVISION'
TT_ERROR_TRANSITION     = 'ERROR_TRANSITION'
TT_ERROR_RECOVERY       = 'ERROR_RECOVERY'
TT_ASSESSMENT_TRANSITION= 'ASSESSMENT_TRANSITION'
TT_NEXT_TASK            = 'NEXT_TASK'
TT_SESSION_RETURN       = 'SESSION_RETURN'

ALL_TRANSITION_TYPES = [
    TT_NEXT_EVENT, TT_RETRY, TT_REVISION,
    TT_ERROR_TRANSITION, TT_ERROR_RECOVERY,
    TT_ASSESSMENT_TRANSITION, TT_NEXT_TASK, TT_SESSION_RETURN,
]

# Node state types derived from event_type
STATE_MAP = {
    'sql_run':       'ATTEMPT',
    'submit_answer': 'ASSESSMENT',
    'sql_error':     'ERROR',
    'hint_request':  'SUPPORT',
    'task_start':    'START',
    'task_end':      'END',
    'session_start': 'START',
    'session_end':   'END',
    # block events reserved — state for future phases
    'block_add':     'BLOCK_ACTION',
    'block_move':    'BLOCK_ACTION',
    'block_delete':  'BLOCK_ACTION',
    'block_submit':  'BLOCK_ACTION',
}

# Anti-leakage: these columns must NEVER appear in graph features
FEATURE_LEAKAGE_COLS = {
    'at_risk', 'total_2c3l_score', 'grade_letter', 'is_teacher_reviewed',
    'c1_correctness_result_score', 'c2_semantic_consistency_score',
    'l1_logical_reasoning_score',  'l2_learning_process_score',
    'l3_difficulty_complexity_score',
}

print(f'Schema version   : {SCHEMA_VERSION}')
print(f'M2 schema source : {M2_SCHEMA_VERSION}')
print(f'TAG dir          : {TAG_DIR}')

Schema version   : tag_v1
M2 schema source : seq_v1
TAG dir          : data\tag


## 1. Load M2 artifacts

In [2]:
manifest_path     = SEQ_DIR / 'sequence_manifest_v1.json'
canonical_path    = SEQ_DIR / 'canonical_events.parquet'
seq_index_path    = SEQ_DIR / 'sequence_index.parquet'
split_ledger_path = SEQ_DIR / 'split_assignments.parquet'

for label, p in [
    ('manifest',      manifest_path),
    ('canonical_events', canonical_path),
    ('sequence_index',   seq_index_path),
    ('split_assignments',split_ledger_path),
]:
    if not p.exists():
        raise FileNotFoundError(
            f'M2 artifact missing: {p}. '
            f'Run 05_sequence_dataset.ipynb first (all 13 checks must pass).'
        )
    print(f'  {label:22s}: {p}')

manifest      = json.loads(manifest_path.read_text())
canonical_df  = pd.read_parquet(canonical_path)
seq_index     = pd.read_parquet(seq_index_path)
split_ledger  = pd.read_parquet(split_ledger_path)

if manifest.get('schema_version') != M2_SCHEMA_VERSION:
    raise ValueError(
        f"M2 manifest schema_version={manifest.get('schema_version')!r}, "
        f"expected {M2_SCHEMA_VERSION!r}. Re-run 05_sequence_dataset.ipynb."
    )

print(f'\nM2 manifest loaded (schema_version={manifest["schema_version"]})')
print(f'canonical_events : {len(canonical_df):,} rows')
print(f'sequence_index   : {len(seq_index):,} rows')
print(f'split_ledger     : {len(split_ledger):,} learners')

  manifest              : data\sequences\sequence_manifest_v1.json
  canonical_events      : data\sequences\canonical_events.parquet
  sequence_index        : data\sequences\sequence_index.parquet
  split_assignments     : data\sequences\split_assignments.parquet



M2 manifest loaded (schema_version=seq_v1)
canonical_events : 702 rows
sequence_index   : 90 rows
split_ledger     : 10 learners


In [3]:
# Derive split and label maps
split_map = dict(zip(split_ledger['academy_member_id'], split_ledger['split']))
label_source_map = dict(zip(split_ledger['academy_member_id'], split_ledger['label_source']))

# Tag each event with split
canonical_df['split'] = canonical_df['academy_member_id'].map(split_map)
canonical_df['event_time'] = pd.to_datetime(canonical_df['event_time'], utc=True, errors='coerce')

# Working set: pre-cutoff, not dropped, has a split assignment
work_df = canonical_df[
    (~canonical_df['dropped_as_duplicate'].fillna(False)) &
    (~canonical_df['is_post_cutoff'].fillna(True)) &
    (canonical_df['split'].notna())
].copy().sort_values(['academy_member_id','task_code','event_order']).reset_index(drop=True)

print(f'\nWorking set (pre-cutoff, deduplicated, in split): {len(work_df):,} events')
print(f'Event types: {sorted(work_df["event_type"].dropna().unique())}')
print(f'Splits: {work_df["split"].value_counts().to_dict()}')


Working set (pre-cutoff, deduplicated, in split): 540 events
Event types: ['sql_error', 'sql_run', 'sql_success']
Splits: {'train': 432, 'test': 108}


## 2. Assign node state types and build node table

In [4]:
def classify_correctness(event_type: str, event_value) -> str:
    if event_type == 'submit_answer':
        val = str(event_value).lower() if pd.notna(event_value) else ''
        if val in ('true', '1', 'correct', 'pass'): return 'CORRECT'
        if val in ('false', '0', 'incorrect', 'fail'): return 'INCORRECT'
        return 'UNKNOWN'
    if event_type == 'sql_run':
        val = str(event_value).lower() if pd.notna(event_value) else ''
        if 'error' in val: return 'ERROR'
        return 'EXECUTED'
    return 'N/A'

def classify_error_state(event_type: str) -> str:
    if event_type == 'sql_error': return 'HAS_ERROR'
    return 'NO_ERROR'

node_records = []
for _, row in work_df.iterrows():
    node_records.append({
        'node_id':          f"{row['academy_member_id']}::{row['task_code']}::{int(row['event_order'])}",
        'sequence_id':      f"{row['academy_member_id']}::{row['task_code']}",
        'academy_member_id':row['academy_member_id'],
        'task_code':        row['task_code'],
        'event_order':      int(row['event_order']),
        'event_timestamp':  row['event_time'],
        'event_type':       row['event_type'],
        'state_type':       STATE_MAP.get(str(row['event_type']), 'OTHER'),
        'correctness_state':classify_correctness(row['event_type'], row.get('event_value')),
        'error_state':      classify_error_state(row['event_type']),
        'elapsed_sec':      float(pd.to_numeric(row.get('duration_from_start'), errors='coerce') or 0.0),
        'is_pre_cutoff':    True,
        'split':            row['split'],
    })

nodes_df = pd.DataFrame(node_records)
print(f'Node table: {len(nodes_df):,} nodes')
print(f'State type distribution:\n{nodes_df["state_type"].value_counts().to_string()}')

Node table: 540 nodes
State type distribution:
state_type
ATTEMPT    270
ERROR      216
OTHER       54


## 3. Classify transition types and build edge table

In [5]:
def classify_transition(
    src_type: str, tgt_type: str,
    delta_sec: float,
    same_session: bool,
    consecutive_run: bool,  # both are sql_run back-to-back (retry)
    src_error: bool,
) -> str:
    """Assign transition type per research contract §6."""
    if not same_session:
        return TT_SESSION_RETURN
    if tgt_type == 'submit_answer':
        return TT_ASSESSMENT_TRANSITION
    if src_type == 'submit_answer' and tgt_type == 'sql_run':
        return TT_REVISION
    if src_error and tgt_type == 'sql_run':
        return TT_ERROR_RECOVERY
    if tgt_type == 'sql_error':
        return TT_ERROR_TRANSITION
    if consecutive_run:
        return TT_RETRY
    return TT_NEXT_EVENT

edge_records = []
edge_counter = 0

for (lid, tc), grp in work_df.groupby(['academy_member_id','task_code'], sort=False):
    grp = grp.sort_values('event_order').reset_index(drop=True)
    seq_id = f'{lid}::{tc}'
    split  = split_map.get(lid, 'unknown')

    for i in range(len(grp) - 1):
        src_row = grp.iloc[i]
        tgt_row = grp.iloc[i + 1]
        src_node = f"{lid}::{tc}::{int(src_row['event_order'])}"
        tgt_node = f"{lid}::{tc}::{int(tgt_row['event_order'])}"

        t_src = src_row['event_time']
        t_tgt = tgt_row['event_time']
        delta_sec = (
            (t_tgt - t_src).total_seconds()
            if pd.notna(t_src) and pd.notna(t_tgt) else 0.0
        )

        same_session  = src_row.get('session_id') == tgt_row.get('session_id')
        consec_run    = (src_row['event_type'] == 'sql_run') and (tgt_row['event_type'] == 'sql_run')
        src_error     = src_row['event_type'] == 'sql_error'

        tt = classify_transition(
            src_row['event_type'], tgt_row['event_type'],
            delta_sec, same_session, consec_run, src_error,
        )

        edge_records.append({
            'edge_id':           f'E{edge_counter:06d}',
            'sequence_id':       seq_id,
            'source_node_id':    src_node,
            'target_node_id':    tgt_node,
            'transition_type':   tt,
            'source_event_type': src_row['event_type'],
            'target_event_type': tgt_row['event_type'],
            'delta_time_sec':    round(delta_sec, 3),
            'transition_order':  i,
            'split':             split,
        })
        edge_counter += 1

edges_df = pd.DataFrame(edge_records)
print(f'Edge table: {len(edges_df):,} edges')
print(f'\nTransition type distribution:')
print(edges_df['transition_type'].value_counts().to_string())

Edge table: 450 edges

Transition type distribution:
transition_type
ERROR_TRANSITION    216
ERROR_RECOVERY      180
NEXT_EVENT           54


## 4. Compute transition statistics

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Transition type counts per split
tt_stats = (
    edges_df.groupby(['split','transition_type'])
    .size().reset_index(name='count')
)
print('Transition counts by split:')
print(tt_stats.to_string(index=False))

# Transition matrix: source_event_type → target_event_type (train only)
train_edges = edges_df[edges_df['split'] == 'train']
transition_matrix = pd.crosstab(
    train_edges['source_event_type'],
    train_edges['target_event_type']
)
print('\nTransition matrix (train):')
print(transition_matrix.to_string())

# Save stats artifact
tt_pivot = (
    edges_df.groupby(['split','transition_type']).size()
    .unstack(fill_value=0).reset_index()
)
for tt in ALL_TRANSITION_TYPES:
    if tt not in tt_pivot.columns:
        tt_pivot[tt] = 0

tt_stats_path = TAG_DIR / 'tag_transition_stats_v1.parquet'
tt_pivot.to_parquet(tt_stats_path, index=False)
print(f'\nTransition stats saved: {tt_stats_path}')

Transition counts by split:
split  transition_type  count
 test   ERROR_RECOVERY     36
 test ERROR_TRANSITION     45
 test       NEXT_EVENT      9
train   ERROR_RECOVERY    144
train ERROR_TRANSITION    171
train       NEXT_EVENT     45

Transition matrix (train):
target_event_type  sql_error  sql_run  sql_success
source_event_type                                 
sql_error                  0      144            0
sql_run                  171        0           45

Transition stats saved: data\tag\tag_transition_stats_v1.parquet


In [7]:
# Research plot: transition type heatmap (train only)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: transition type bar chart
ax = axes[0]
tc_counts = edges_df['transition_type'].value_counts()
ax.barh(tc_counts.index, tc_counts.values, edgecolor='black')
ax.set_xlabel('Count');  ax.set_title('Transition types (all splits)')
ax.invert_yaxis()

# Right: transition matrix heatmap (train only)
ax2 = axes[1]
if not transition_matrix.empty:
    im = ax2.imshow(transition_matrix.values, aspect='auto', cmap='Blues')
    ax2.set_xticks(range(len(transition_matrix.columns)))
    ax2.set_yticks(range(len(transition_matrix.index)))
    ax2.set_xticklabels(transition_matrix.columns, rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(transition_matrix.index, fontsize=8)
    plt.colorbar(im, ax=ax2)
ax2.set_title('Event-type transition matrix (train)')
ax2.set_xlabel('Target event type');  ax2.set_ylabel('Source event type')

plt.tight_layout()
plt.savefig(REP_DIR / 'tag_transition_heatmap.png', dpi=150);  plt.close()
print(f'Saved: {REP_DIR}/tag_transition_heatmap.png')

Saved: ..\reports\phase4/tag_transition_heatmap.png


## 5. Student-instance TAGs

One directed graph per learner × task sequence.  
Used for learning-path analysis and to validate edge classification.

In [8]:
def build_instance_tag(sequence_id: str, nodes: pd.DataFrame, edges: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()
    G.graph['sequence_id'] = sequence_id
    for _, row in nodes.iterrows():
        G.add_node(row['node_id'],
                   event_type=row['event_type'],
                   state_type=row['state_type'],
                   correctness_state=row['correctness_state'],
                   error_state=row['error_state'],
                   elapsed_sec=row['elapsed_sec'],
                   event_order=row['event_order'])
    for _, row in edges.iterrows():
        G.add_edge(row['source_node_id'], row['target_node_id'],
                   edge_id=row['edge_id'],
                   transition_type=row['transition_type'],
                   delta_time_sec=row['delta_time_sec'])
    return G

node_by_seq  = {sid: grp for sid, grp in nodes_df.groupby('sequence_id')}
edge_by_seq  = {sid: grp for sid, grp in edges_df.groupby('sequence_id')}
all_seq_ids  = sorted(set(nodes_df['sequence_id'].unique()) | set(edges_df['sequence_id'].unique()))

instance_tags = {}
for sid in all_seq_ids:
    n = node_by_seq.get(sid, pd.DataFrame())
    e = edge_by_seq.get(sid, pd.DataFrame())
    instance_tags[sid] = build_instance_tag(sid, n, e)

print(f'Student-instance TAGs built: {len(instance_tags)}')
if instance_tags:
    sample_id   = all_seq_ids[0]
    sample_tag  = instance_tags[sample_id]
    print(f'\nSample TAG — {sample_id}')
    print(f'  Nodes : {sample_tag.number_of_nodes()}')
    print(f'  Edges : {sample_tag.number_of_edges()}')

Student-instance TAGs built: 90

Sample TAG — MOCK_VALID3_20260715_S001::LQT000001
  Nodes : 6
  Edges : 5


In [9]:
# Compute per-sequence stats for the sequence stats artifact
seq_stat_records = []
for sid, G in instance_tags.items():
    lid, tc = sid.split('::', 1)
    e_types = [d['event_type'] for _, d in G.nodes(data=True)]
    t_types = [d['transition_type'] for _, _, d in G.edges(data=True)]
    etypes_counter = Counter(e_types)
    ttypes_counter = Counter(t_types)
    n_nodes  = G.number_of_nodes()
    n_edges  = G.number_of_edges()
    seq_stat_records.append({
        'sequence_id':         sid,
        'academy_member_id':   lid,
        'task_code':           tc,
        'split':               split_map.get(lid, 'unknown'),
        'n_nodes':             n_nodes,
        'n_edges':             n_edges,
        'n_unique_event_types':len(etypes_counter),
        'n_sql_run':           etypes_counter.get('sql_run', 0),
        'n_submit_answer':     etypes_counter.get('submit_answer', 0),
        'n_sql_error':         etypes_counter.get('sql_error', 0),
        'n_hint_request':      etypes_counter.get('hint_request', 0),
        'n_retry':             ttypes_counter.get(TT_RETRY, 0),
        'n_revision':          ttypes_counter.get(TT_REVISION, 0),
        'n_error_transition':  ttypes_counter.get(TT_ERROR_TRANSITION, 0),
        'n_error_recovery':    ttypes_counter.get(TT_ERROR_RECOVERY, 0),
        'n_assessment':        ttypes_counter.get(TT_ASSESSMENT_TRANSITION, 0),
        'n_session_return':    ttypes_counter.get(TT_SESSION_RETURN, 0),
        'is_dag':              nx.is_directed_acyclic_graph(G),
    })

seq_stats_df = pd.DataFrame(seq_stat_records)
seq_stats_path = TAG_DIR / 'tag_sequence_stats_v1.parquet'
seq_stats_df.to_parquet(seq_stats_path, index=False)
print(f'Sequence stats saved: {seq_stats_path}  ({len(seq_stats_df)} rows)')
display(seq_stats_df.describe(include='number').T)

Sequence stats saved: data\tag\tag_sequence_stats_v1.parquet  (90 rows)


,count,mean,std,min,25%,50%,75%,max
n_nodes,90.0,6.0,0.000000,6.0,6.0,6.0,6.0,6.0
n_edges,90.0,5.0,0.000000,5.0,5.0,5.0,5.0,5.0
n_unique_event_types,90.0,2.6,0.492642,2.0,2.0,3.0,3.0,3.0
n_sql_run,90.0,3.0,0.000000,3.0,3.0,3.0,3.0,3.0
n_submit_answer,90.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
n_sql_error,90.0,2.4,0.492642,2.0,2.0,2.0,3.0,3.0
n_hint_request,90.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
n_retry,90.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
n_revision,90.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
n_error_transition,90.0,2.4,0.492642,2.0,2.0,2.0,3.0,3.0


## 6. Cohort Transition TAG

Two cohort graphs are built:
- **Train-cohort TAG** — used for predictive analysis and graph feature statistics
- **Full-pilot TAG** — train + test combined, for descriptive analysis only

Test split events are never aggregated into the train-cohort TAG.

In [10]:
def build_cohort_tag(edges_subset: pd.DataFrame, label: str) -> nx.DiGraph:
    """Aggregate individual transitions into a weighted cohort graph."""
    G = nx.DiGraph()
    G.graph['label'] = label
    for _, row in edges_subset.iterrows():
        src = row['source_event_type']
        tgt = row['target_event_type']
        tt  = row['transition_type']
        if G.has_edge(src, tgt):
            G[src][tgt]['weight'] += 1
            G[src][tgt]['transition_types'][tt] = G[src][tgt]['transition_types'].get(tt, 0) + 1
        else:
            G.add_edge(src, tgt, weight=1,
                       dominant_transition=tt,
                       transition_types={tt: 1})
    for u, v, data in G.edges(data=True):
        tt_counts = data['transition_types']
        G[u][v]['dominant_transition'] = max(tt_counts, key=tt_counts.get)
    return G

train_cohort_tag = build_cohort_tag(edges_df[edges_df['split'] == 'train'], 'train_cohort')
full_pilot_tag   = build_cohort_tag(edges_df, 'full_pilot_descriptive')

print(f'Train-cohort TAG  — nodes:{train_cohort_tag.number_of_nodes()}  edges:{train_cohort_tag.number_of_edges()}')
print(f'Full-pilot TAG    — nodes:{full_pilot_tag.number_of_nodes()}    edges:{full_pilot_tag.number_of_edges()}')
print()
print('Train-cohort edge weights (source → target  weight  dominant_transition):')
for u, v, data in sorted(train_cohort_tag.edges(data=True), key=lambda x: -x[2]['weight']):
    print(f'  {u:<20} → {v:<20}  w={data["weight"]:3d}  {data["dominant_transition"]}')

Train-cohort TAG  — nodes:3  edges:3
Full-pilot TAG    — nodes:3    edges:3

Train-cohort edge weights (source → target  weight  dominant_transition):
  sql_run              → sql_error             w=171  ERROR_TRANSITION
  sql_error            → sql_run               w=144  ERROR_RECOVERY
  sql_run              → sql_success           w= 45  NEXT_EVENT


In [11]:
# Visualize train-cohort TAG
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, G, title in [
    (axes[0], train_cohort_tag, 'Train-cohort TAG'),
    (axes[1], full_pilot_tag,   'Full-pilot TAG (descriptive only)'),
]:
    if G.number_of_nodes() == 0:
        ax.set_title(f'{title} — empty graph')
        continue
    try:
        pos    = nx.spring_layout(G, seed=42, k=2)
        weights= [G[u][v]['weight'] for u, v in G.edges()]
        max_w  = max(weights) if weights else 1
        widths = [1 + 4 * w / max_w for w in weights]
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=600,
                               node_color='lightsteelblue', edgecolors='steelblue')
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
        nx.draw_networkx_edges(G, pos, ax=ax, width=widths, edge_color='steelblue',
                               arrows=True, arrowsize=15,
                               connectionstyle='arc3,rad=0.1')
        edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                     ax=ax, font_size=6)
    except Exception as exc:
        ax.text(0.5, 0.5, str(exc), ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title);  ax.axis('off')

plt.tight_layout()
plt.savefig(REP_DIR / 'tag_cohort_graphs.png', dpi=150);  plt.close()
print(f'Saved: {REP_DIR}/tag_cohort_graphs.png')

Saved: ..\reports\phase4/tag_cohort_graphs.png


## 7. Graph-derived features

18 structural features per `sequence_id`.  
**All features are pre-cutoff only. No outcome data.**

In [12]:
FEATURE_NAMES = [
    'node_count',
    'edge_count',
    'unique_event_types',
    'retry_count',
    'revision_count',
    'error_transition_count',
    'error_recovery_count',
    'error_recovery_rate',
    'assessment_count',
    'session_return_count',
    'transition_entropy',
    'event_type_entropy',
    'mean_delta_time_sec',
    'std_delta_time_sec',
    'max_delta_time_sec',
    'min_delta_time_sec',
    'run_to_submit_ratio',
    'graph_density',
]

def compute_graph_features(sequence_id: str, G: nx.DiGraph, seq_edges: pd.DataFrame) -> dict:
    node_count   = G.number_of_nodes()
    edge_count   = G.number_of_edges()

    event_types  = [G.nodes[n]['event_type'] for n in G.nodes]
    et_counter   = Counter(event_types)
    unique_et    = len(et_counter)

    tt_vals = seq_edges['transition_type'].tolist() if not seq_edges.empty else []
    tt_counter = Counter(tt_vals)

    retry_count    = tt_counter.get(TT_RETRY, 0)
    revision_count = tt_counter.get(TT_REVISION, 0)
    err_trans      = tt_counter.get(TT_ERROR_TRANSITION, 0)
    err_rec        = tt_counter.get(TT_ERROR_RECOVERY, 0)
    assessment_c   = tt_counter.get(TT_ASSESSMENT_TRANSITION, 0)
    sess_ret       = tt_counter.get(TT_SESSION_RETURN, 0)

    err_rec_rate   = err_rec / (err_trans + 1e-9) if err_trans > 0 else 0.0

    # Transition type entropy
    if tt_counter:
        tt_arr = np.array(list(tt_counter.values()), dtype=float)
        tt_entropy = float(scipy_entropy(tt_arr / tt_arr.sum() + 1e-12))
    else:
        tt_entropy = 0.0

    # Event type entropy
    if et_counter:
        et_arr = np.array(list(et_counter.values()), dtype=float)
        et_entropy = float(scipy_entropy(et_arr / et_arr.sum() + 1e-12))
    else:
        et_entropy = 0.0

    # Delta time statistics
    delta_times = seq_edges['delta_time_sec'].dropna().values if not seq_edges.empty else np.array([])
    delta_times = delta_times[delta_times >= 0]  # guard against clock skew
    mean_dt = float(np.mean(delta_times)) if len(delta_times) > 0 else 0.0
    std_dt  = float(np.std(delta_times))  if len(delta_times) > 1 else 0.0
    max_dt  = float(np.max(delta_times))  if len(delta_times) > 0 else 0.0
    min_dt  = float(np.min(delta_times))  if len(delta_times) > 0 else 0.0

    # Run-to-submit ratio
    n_run    = et_counter.get('sql_run', 0)
    n_submit = et_counter.get('submit_answer', 0)
    run_to_sub = n_run / (n_submit + 1e-9) if n_submit > 0 else float(n_run)

    # Graph density (directed)
    max_edges  = node_count * (node_count - 1) if node_count > 1 else 1
    density    = edge_count / max_edges

    return {
        'sequence_id':          sequence_id,
        'node_count':           node_count,
        'edge_count':           edge_count,
        'unique_event_types':   unique_et,
        'retry_count':          retry_count,
        'revision_count':       revision_count,
        'error_transition_count': err_trans,
        'error_recovery_count': err_rec,
        'error_recovery_rate':  round(err_rec_rate, 6),
        'assessment_count':     assessment_c,
        'session_return_count': sess_ret,
        'transition_entropy':   round(tt_entropy, 6),
        'event_type_entropy':   round(et_entropy, 6),
        'mean_delta_time_sec':  round(mean_dt, 3),
        'std_delta_time_sec':   round(std_dt, 3),
        'max_delta_time_sec':   round(max_dt, 3),
        'min_delta_time_sec':   round(min_dt, 3),
        'run_to_submit_ratio':  round(run_to_sub, 6),
        'graph_density':        round(density, 6),
    }

feat_records = []
for sid in all_seq_ids:
    G          = instance_tags[sid]
    seq_edges  = edge_by_seq.get(sid, pd.DataFrame())
    feat_records.append(compute_graph_features(sid, G, seq_edges))

feat_df = pd.DataFrame(feat_records)
# Merge split from nodes
feat_df = feat_df.merge(
    nodes_df[['sequence_id','split','academy_member_id','task_code']].drop_duplicates('sequence_id'),
    on='sequence_id', how='left',
)
print(f'Graph-derived features: {len(feat_df)} rows × {len(FEATURE_NAMES)+1} feature cols')
display(feat_df.describe(include='number').T)

Graph-derived features: 90 rows × 19 feature cols


,count,mean,std,min,25%,50%,75%,max
node_count,90.0,6.000000,0.000000e+00,6.000000,6.000000,6.000000,6.000000,6.000000
edge_count,90.0,5.000000,0.000000e+00,5.000000,5.000000,5.000000,5.000000,5.000000
unique_event_types,90.0,2.600000,4.926425e-01,2.000000,2.000000,3.000000,3.000000,3.000000
retry_count,90.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
revision_count,90.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
error_transition_count,90.0,2.400000,4.926425e-01,2.000000,2.000000,2.000000,3.000000,3.000000
error_recovery_count,90.0,2.000000,0.000000e+00,2.000000,2.000000,2.000000,2.000000,2.000000
error_recovery_rate,90.0,0.866667,1.642140e-01,0.666667,0.666667,1.000000,1.000000,1.000000
assessment_count,90.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
session_return_count,90.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000


In [13]:
# Verify no leakage column slipped into feature table
leaked_cols = sorted(set(feat_df.columns) & FEATURE_LEAKAGE_COLS)
if leaked_cols:
    raise ValueError(f'LEAKAGE in graph features: {leaked_cols}')
print('Anti-leakage check (feature table): PASS — no blacklisted columns')

nan_cols = feat_df[FEATURE_NAMES].columns[feat_df[FEATURE_NAMES].isna().any()].tolist()
if nan_cols:
    raise ValueError(f'NaN in graph feature columns: {nan_cols}')
print('NaN check (feature table)         : PASS — no NaN in feature columns')

feat_path = TAG_DIR / 'tag_graph_features_v1.parquet'
feat_df.to_parquet(feat_path, index=False)
print(f'\nGraph features saved: {feat_path}')

Anti-leakage check (feature table): PASS — no blacklisted columns
NaN check (feature table)         : PASS — no NaN in feature columns

Graph features saved: data\tag\tag_graph_features_v1.parquet


In [14]:
# Research plot: feature distribution overview
plot_cols = [
    'node_count','edge_count','retry_count','error_recovery_rate',
    'transition_entropy','mean_delta_time_sec','run_to_submit_ratio','graph_density',
]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, plot_cols):
    for split_name, color in [('train','steelblue'), ('test','darkorange')]:
        vals = feat_df[feat_df['split'] == split_name][col].dropna().values
        if len(vals) > 0:
            ax.hist(vals, bins=10, alpha=0.6, color=color, label=split_name, edgecolor='white')
    ax.set_title(col, fontsize=8);  ax.legend(fontsize=7)
plt.suptitle('Graph-derived feature distributions (train=blue, test=orange)', fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(REP_DIR / 'tag_feature_distributions.png', dpi=150);  plt.close()
print(f'Saved: {REP_DIR}/tag_feature_distributions.png')

Saved: ..\reports\phase4/tag_feature_distributions.png


## 8. Save node and edge artifacts

In [15]:
nodes_path = TAG_DIR / 'tag_nodes_v1.parquet'
edges_path = TAG_DIR / 'tag_edges_v1.parquet'

nodes_df.to_parquet(nodes_path, index=False)
edges_df.to_parquet(edges_path, index=False)

print(f'Nodes saved : {nodes_path}  ({len(nodes_df):,} rows)')
print(f'Edges saved : {edges_path}  ({len(edges_df):,} rows)')
print(f'Seq stats   : {seq_stats_path}  ({len(seq_stats_df)} rows)')
print(f'Trans stats : {tt_stats_path}')
print(f'Features    : {feat_path}  ({len(feat_df)} rows)')

Nodes saved : data\tag\tag_nodes_v1.parquet  (540 rows)
Edges saved : data\tag\tag_edges_v1.parquet  (450 rows)
Seq stats   : data\tag\tag_sequence_stats_v1.parquet  (90 rows)
Trans stats : data\tag\tag_transition_stats_v1.parquet
Features    : data\tag\tag_graph_features_v1.parquet  (90 rows)


## 9. Write TAG manifest

In [16]:
def sha256_parquet(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

tag_manifest = {
    'schema_version':    SCHEMA_VERSION,
    'created_at_utc':    datetime.now(timezone.utc).isoformat(),
    'phase3_source_sha': PHASE3_SOURCE_SHA,
    'm2_manifest_sha':   sha256_parquet(manifest_path),
    'transition_types':  ALL_TRANSITION_TYPES,
    'graph_feature_names': FEATURE_NAMES,
    'n_features':        len(FEATURE_NAMES),
    'dataset_stats': {
        'total_sequences':        len(all_seq_ids),
        'total_nodes':            int(len(nodes_df)),
        'total_edges':            int(len(edges_df)),
        'train_sequences':        int((feat_df['split'] == 'train').sum()),
        'test_sequences':         int((feat_df['split'] == 'test').sum()),
        'train_cohort_nodes':     train_cohort_tag.number_of_nodes(),
        'train_cohort_edges':     train_cohort_tag.number_of_edges(),
        'full_pilot_nodes':       full_pilot_tag.number_of_nodes(),
        'full_pilot_edges':       full_pilot_tag.number_of_edges(),
        'feature_leakage_check':  'PASS',
        'nan_in_features':        'NONE',
    },
    'artifacts': {
        'tag_nodes':          str(nodes_path),
        'tag_edges':          str(edges_path),
        'tag_transition_stats': str(tt_stats_path),
        'tag_sequence_stats': str(seq_stats_path),
        'tag_graph_features': str(feat_path),
    },
    'artifact_checksums': {
        'tag_nodes':            sha256_parquet(nodes_path),
        'tag_edges':            sha256_parquet(edges_path),
        'tag_transition_stats': sha256_parquet(tt_stats_path),
        'tag_sequence_stats':   sha256_parquet(seq_stats_path),
        'tag_graph_features':   sha256_parquet(feat_path),
    },
    'data_warning': (
        'TECHNICAL VALIDATION ONLY — mock dataset, proxy_behavioral/pilot_only labels. '
        'TAG is not a classifier in Phase 4. Do not report AUC/F1. '
        'Final thesis requires teacher-reviewed labels and >=60 participants.'
    ),
}

tag_manifest_path = TAG_DIR / 'tag_manifest_v1.json'
tag_manifest_path.write_text(json.dumps(tag_manifest, indent=2, default=str))
print(f'TAG manifest saved: {tag_manifest_path}')
print(json.dumps(tag_manifest['dataset_stats'], indent=2))

TAG manifest saved: data\tag\tag_manifest_v1.json
{
  "total_sequences": 90,
  "total_nodes": 540,
  "total_edges": 450,
  "train_sequences": 72,
  "test_sequences": 18,
  "train_cohort_nodes": 3,
  "train_cohort_edges": 3,
  "full_pilot_nodes": 3,
  "full_pilot_edges": 3,
  "feature_leakage_check": "PASS",
  "nan_in_features": "NONE"
}


## 10. M3 Validation — 12 checks

In [17]:
checks = []
def chk(name: str, passed: bool, detail: str = ''):
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})

# 1. M2 artifacts loaded
chk('M2 artifacts loaded (4 files)',
    all([canonical_path.exists(), seq_index_path.exists(),
         split_ledger_path.exists(), manifest_path.exists()]))

# 2. Schema version match
chk('M2 schema version matches',
    manifest.get('schema_version') == M2_SCHEMA_VERSION,
    f"found: {manifest.get('schema_version')}")

# 3. Node table schema
req_node_cols = {'node_id','sequence_id','academy_member_id','task_code',
                 'event_order','event_type','state_type','correctness_state',
                 'error_state','elapsed_sec','is_pre_cutoff','split'}
missing_node  = sorted(req_node_cols - set(nodes_df.columns))
chk('Node table has required schema', len(missing_node) == 0,
    f'missing: {missing_node}' if missing_node else f'{len(nodes_df)} rows')

# 4. Edge table schema
req_edge_cols = {'edge_id','sequence_id','source_node_id','target_node_id',
                 'transition_type','source_event_type','target_event_type',
                 'delta_time_sec','transition_order','split'}
missing_edge  = sorted(req_edge_cols - set(edges_df.columns))
chk('Edge table has required schema', len(missing_edge) == 0,
    f'missing: {missing_edge}' if missing_edge else f'{len(edges_df)} rows')

# 5. All transition types are valid
invalid_tt = sorted(set(edges_df['transition_type'].unique()) - set(ALL_TRANSITION_TYPES))
chk('All transition types are valid', len(invalid_tt) == 0,
    f'invalid types: {invalid_tt}' if invalid_tt else 'all valid')

# 6. No post-cutoff events in nodes
chk('No post-cutoff events in node table',
    nodes_df['is_pre_cutoff'].all(),
    f'all is_pre_cutoff=True')

# 7. Anti-leakage in feature table
chk('Anti-leakage in graph features', len(leaked_cols) == 0,
    f'leaked: {leaked_cols}' if leaked_cols else 'no blacklisted columns')

# 8. No NaN in feature table
chk('No NaN in graph feature columns', len(nan_cols) == 0,
    f'NaN in: {nan_cols}' if nan_cols else 'clean')

# 9. Feature table row count matches sequence count
chk('Feature rows == sequence count',
    len(feat_df) == len(all_seq_ids),
    f'feat_df={len(feat_df)}  all_seq_ids={len(all_seq_ids)}')

# 10. Train/test split preserved in feature table
split_vals = set(feat_df['split'].dropna().unique())
chk('Train/test splits present in features',
    'train' in split_vals,
    f'splits found: {sorted(split_vals)}')

# 11. All 6 artifact files exist
artifact_files = [nodes_path, edges_path, tt_stats_path, seq_stats_path, feat_path, tag_manifest_path]
missing_files  = [str(f) for f in artifact_files if not f.exists()]
chk('All 6 TAG artifacts exist', len(missing_files) == 0,
    f'missing: {missing_files}' if missing_files else '6/6 files present')

# 12. TAG manifest contains checksums
has_checksums = 'artifact_checksums' in tag_manifest and len(tag_manifest['artifact_checksums']) == 5
chk('TAG manifest contains 5 checksums', has_checksums,
    f'{len(tag_manifest.get("artifact_checksums", {}))} checksums')

# ── Summary ──────────────────────────────────────────────────────────────────
result_df = pd.DataFrame(checks)
n_fail    = (result_df['result'] == 'FAIL').sum()

print('\n── M3 Validation Summary ─────────────────────────────────────────')
print(result_df.to_string(index=False))
print(f'\n{len(checks) - n_fail}/{len(checks)} checks passed')

if n_fail > 0:
    raise RuntimeError(f'M3 validation FAILED — {n_fail} check(s) did not pass.')

print('\n✅ M3 COMPLETE — TAG artifacts ready for M4 (LSTM/GRU).')
print('\n⚠️  TECHNICAL VALIDATION ONLY.')
print(f'   Sequences={len(all_seq_ids)}  Nodes={len(nodes_df)}  Edges={len(edges_df)}')
print(f'   Graph features: {len(FEATURE_NAMES)} dimensions per sequence')
print(f'   TAG is NOT a classifier — no AUC/F1 reported in M3.')
print(f'   Final thesis requires teacher-reviewed labels and >=60 participants.')


── M3 Validation Summary ─────────────────────────────────────────
                                check result                          detail
        M2 artifacts loaded (4 files)   PASS                                
            M2 schema version matches   PASS                   found: seq_v1
       Node table has required schema   PASS                        540 rows
       Edge table has required schema   PASS                        450 rows
       All transition types are valid   PASS                       all valid
  No post-cutoff events in node table   PASS          all is_pre_cutoff=True
       Anti-leakage in graph features   PASS          no blacklisted columns
      No NaN in graph feature columns   PASS                           clean
       Feature rows == sequence count   PASS      feat_df=90  all_seq_ids=90
Train/test splits present in features   PASS splits found: ['test', 'train']
            All 6 TAG artifacts exist   PASS               6/6 files present
    TAG 